# OddsJam EV Filter Research — Notebook 02: Filter Evaluation and Exchange Diagnostics

## Purpose

Evaluate historical executed-bet performance across saved filters and venue regimes using canonical evaluation tables, exchange liquidity-capacity diagnostics, and scorecards. These are descriptive research outputs, not validated filter recommendations.

**Input:** processed Parquet snapshot produced by `scripts/preprocess_bet_tracker.py`. Run from `notebooks/` with the project Poetry kernel. Set `AS_OF_DATE` to an existing processed snapshot suffix.

**Interpretation:** historical filter labels may not capture configuration changes; repeated executions and overlapping opportunities are not independent bets; modeled EV is reference-dependent; realized ROI is noisy; exchange liquidity buckets are incomplete. Future strategy changes require out-of-time validation.

**Workflow:** first verify Notebook 01 data readiness for the same snapshot, then execute this notebook. Subsequent visualizations and expanded segment analyses can build on these canonical tables.


## Visual research and interpretation

This notebook also contains the executed-bet performance charts migrated from the data-readiness notebook. All plots operate on the independently loaded processed snapshot. Historical ROI, modeled EV, and CLV are different measures; observed segment differences are **hypotheses** until checked for sample size, overlap, execution bias, and out-of-time stability.


### Reproducible snapshot selection

Dataset and artifact paths are resolved by `resolve_snapshot_config()`. Set `ODDSJAM_SNAPSHOT_DATE=MM-DD-YYYY` before launching the kernel to select a different processed export; the default remains an explicit historical snapshot. Both notebooks can run from any working directory when the project package is installed.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oddsjam_ev.analysis.candidate_review import CandidateReviewConfig, build_candidate_review
from oddsjam_ev.analysis.exchange_eval import build_exchange_capacity_table
from oddsjam_ev.analysis.filter_eval import FilterEvalConfig, build_filter_evaluation_table
from oddsjam_ev.analysis.research_grains import build_research_grains
from oddsjam_ev.analysis.scorecards import ExchangeScorecardConfig, build_exchange_scorecard
from oddsjam_ev.analysis.segment_eval import SegmentEvalConfig, build_segment_evaluation
from oddsjam_ev.analysis.snapshot_config import resolve_snapshot_config
from oddsjam_ev.analysis.temporal_validation import (
    TemporalValidationConfig,
    build_group_aware_temporal_split,
)
from oddsjam_ev.plotting import (
    AvgBetsByTimeBucketPlotConfig,
    BetCountOverTimePlotConfig,
    CombinedEVPlotConfig,
    CumProfitByGroupPlotConfig,
    ProfitByGroupPlotConfig,
    combined_ev_summary_plot,
    combined_ev_summary_plot_faceted_by_group,
    plot_avg_bets_by_time_bucket,
    plot_bet_count_over_time,
    plot_cum_profit_by_group,
    plot_profit_by_group_for_filters,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
snapshot_cfg = resolve_snapshot_config()
AS_OF_DATE = snapshot_cfg.snapshot_date
ARTIFACT_DIR = snapshot_cfg.artifact_dir
WRITE_ARTIFACTS = False
processed_path = snapshot_cfg.processed_path
if not processed_path.is_file():
    raise FileNotFoundError(
        f"Processed snapshot not found: {processed_path}. "
        "Run preprocessing or select an existing ODDSJAM_SNAPSHOT_DATE."
    )
df_filtered = pd.read_parquet(processed_path)
print(f"Loaded {len(df_filtered):,} processed rows from {processed_path}")
print(f"Repository root: {snapshot_cfg.project_root}; snapshot: {AS_OF_DATE}")

## Execution and opportunity-group research populations

The processed Bet Tracker dataset is **execution-grain**: each row represents a recorded bet, not necessarily a unique market opportunity. `execution_rows` preserves every processed record and adds a dataset-local row identifier. It remains the appropriate input for realized profit, stake, and ROI accounting.

`opportunity_groups` summarizes rows sharing the existing preprocessing `opportunity_id`. This is a **grouping audit**, not an independently verified quote-level opportunity dataset. Repeated rows may be legitimate multiple executions, not export duplicates. Missing IDs are kept as distinct surrogate groups. Group-level stake and profit sums are reconciliation diagnostics; do **not** compute independent-bet ROI, statistical significance, or effective sample size from these groups without validating the grouping definition.

A time-based holdout should keep all executions with the same research opportunity key in the same split. Source opportunity identifiers should be audited for stability across exports and for collisions before being treated as durable keys. Saved-filter membership can overlap and is not automatically a unique opportunity dimension.


In [ ]:
# Construct explicit research populations without modifying the existing evaluation inputs.
execution_rows, opportunity_groups = build_research_grains(df_filtered)

assert len(execution_rows) == len(df_filtered)
assert int(opportunity_groups["execution_count"].sum()) == len(execution_rows)

n_missing_ids = int(execution_rows["opportunity_id_missing"].sum())
n_repeated_groups = int(opportunity_groups["repeated_execution_group"].sum())
n_repeated_executions = int(
    opportunity_groups.loc[opportunity_groups["repeated_execution_group"], "execution_count"].sum()
)

print(f"Execution rows: {len(execution_rows):,}")
print(f"Opportunity groups (proxy): {len(opportunity_groups):,}")
print(f"Executions missing source opportunity ID: {n_missing_ids:,}")
print(f"Groups with multiple executions: {n_repeated_groups:,}")
print(f"Executions in repeated groups: {n_repeated_executions:,}")
print("Note: repeated groups are not automatically duplicate errors.")
display(
    opportunity_groups["execution_count"]
    .value_counts()
    .sort_index()
    .head(15)
    .rename_axis("executions_per_group")
    .to_frame("group_count")
)
display(opportunity_groups.sort_values("execution_count", ascending=False).head(10))

## Descriptive segment evaluation — initial engine smoke test

The reusable segment engine summarizes **execution-level** records by one dimension, with a minimum-count flag and opportunity-group counts. These outputs are exploratory and do not replace canonical filter evaluation. In particular, segments are **not** independent observations and the sample-size flag does not imply statistical significance.

This section checks the engine against the loaded snapshot. Full filter-level decomposition, predeclared thresholds, time splits, and sensitivity analyses belong in the subsequent research workflow. The module supports explicitly specified numeric bins and modeled EV *amounts* when their source-column semantics have been verified.

In [ ]:
# Use a common categorical dimension; avoid assuming optional fields exist.
segment_dimension = next(
    (col for col in ("sportsbook", "sport", "regime") if col in execution_rows.columns),
    None,
)
if segment_dimension is None:
    print("No supported categorical segment dimension is present in this snapshot.")
else:
    segment_cfg = SegmentEvalConfig(min_executions=100)
    segment_preview = build_segment_evaluation(
        execution_rows, dimension=segment_dimension, cfg=segment_cfg
    )
    print(f"Descriptive segment evaluation by {segment_dimension}:")
    display(segment_preview.sort_values("execution_count", ascending=False).head(20))
    print("Minimum sample flags are descriptive, not significance tests.")

## Exploratory segment decomposition and temporal stability

The tables below apply the reusable segment evaluator to **one dimension at a time**. They describe recorded executions, not all opportunities displayed by OddsJam. Filters can overlap, and repeated opportunity groups can induce dependence. A minimum execution count is a screening flag, **not** statistical significance.

The date split uses chronological halves of *observed executions*, not an independently reserved holdout. No segment is selected or optimized on the later half. Changes in realized ROI may reflect sampling noise, seasonality, execution mix, or market conditions. Numeric ranges are specified in advance and reported only when their source fields can be identified unambiguously.

All ROI values are fractions; 0.05 corresponds to 5%. Segment totals reconcile within each dimension but must not be summed across dimensions.

In [ ]:
# Descriptive, one-dimensional segment comparisons with explicit coverage checks.
SEGMENT_MIN_EXECUTIONS = 100
SEGMENT_DIMENSIONS = (
    "regime",
    "sportsbook",
    "sport",
    "league",
    "market",
    "saved_filter_names",
)
segment_cfg = SegmentEvalConfig(min_executions=SEGMENT_MIN_EXECUTIONS)
segment_tables = {}
segment_coverage = []
for dimension in SEGMENT_DIMENSIONS:
    if dimension not in execution_rows.columns:
        segment_coverage.append({"dimension": dimension, "status": "missing column"})
        continue
    table = build_segment_evaluation(execution_rows, dimension, cfg=segment_cfg)
    assert int(table["execution_count"].sum()) == len(execution_rows), dimension
    segment_tables[dimension] = table
    segment_coverage.append(
        {
            "dimension": dimension,
            "status": "evaluated",
            "segments": len(table),
            "small_segments": int((~table["meets_min_executions"]).sum()),
        }
    )
print("Segment availability and sample sufficiency:")
display(pd.DataFrame(segment_coverage))
for dimension, table in segment_tables.items():
    print(f"{dimension}: largest segments by execution count (descriptive)")
    display(table.sort_values("execution_count", ascending=False).head(12))

### Predefined numeric ranges

Numeric dimensions are evaluated only if a recognizable, numeric source column is available. These bins are exploratory and do not represent the current OddsJam UI configuration. In particular, a liquidity bucket is not necessarily a verified executable amount, and a displayed EV percentage is not a realized return. Ambiguous or unavailable fields are reported rather than silently substituted.

In [ ]:
# Explicit candidate columns and fixed research bins; skip ambiguous matches.
NUMERIC_SPECS = {
    "odds": (("american_odds", "bet_odds", "odds"), [-10000, -300, -150, -110, 0, 150, 300, 10000]),
    "ev_pct": (("ev_pct", "ev_percentage", "expected_value_pct"), [-100, 0, 2.5, 5, 10, 20, 1000]),
    "liquidity": (
        ("liquidity", "available_liquidity", "liquidity_amount"),
        [0, 50, 100, 300, 500, 1000, float("inf")],
    ),
}
numeric_coverage = []
for label, (candidates, edges) in NUMERIC_SPECS.items():
    present = [col for col in candidates if col in execution_rows.columns]
    if len(present) != 1:
        numeric_coverage.append(
            {
                "dimension": label,
                "status": "ambiguous" if present else "unavailable",
                "matching_columns": ", ".join(present),
            }
        )
        continue
    column = present[0]
    values = pd.to_numeric(execution_rows[column], errors="coerce")
    if not values.notna().any():
        numeric_coverage.append(
            {"dimension": label, "status": "no numeric values", "matching_columns": column}
        )
        continue
    table = build_segment_evaluation(execution_rows, column, cfg=segment_cfg, bin_edges=edges)
    assert int(table["execution_count"].sum()) == len(execution_rows)
    numeric_coverage.append(
        {
            "dimension": label,
            "status": "evaluated",
            "matching_columns": column,
            "numeric_coverage_pct": round(100 * values.notna().mean(), 1),
        }
    )
    print(f"{label} ({column}): fixed-bin descriptive results")
    display(table)
display(pd.DataFrame(numeric_coverage))

### Chronological stability check

Where a recognizable execution timestamp exists, split the observed executions at their median valid timestamp and compare **the same predefined segments** across earlier and later periods. Missing timestamps are excluded from this particular comparison and their count is reported. Each period is evaluated independently, so sparse-period flags are retained. This is an exploratory stability diagnostic, **not** a prospective holdout or evidence of a causal filter improvement.

In [ ]:
# Do not infer a timestamp from event start time or use a date with unclear semantics.
TIMESTAMP_CANDIDATES = (
    "bet_time",
    "bet_datetime",
    "bet_timestamp",
    "placed_at",
    "timestamp",
    "bet_ts",
)
present_ts = [col for col in TIMESTAMP_CANDIDATES if col in execution_rows.columns]
if len(present_ts) != 1:
    print(
        f"Chronological comparison not assessed: expected one execution timestamp; found {present_ts}."
    )
else:
    ts_column = present_ts[0]
    ts = pd.to_datetime(execution_rows[ts_column], errors="coerce", utc=True)
    valid = ts.notna()
    print(
        f"Execution timestamp field: {ts_column}; valid={int(valid.sum()):,}; missing={int((~valid).sum()):,}"
    )
    if valid.sum() < 2 or ts[valid].nunique() < 2:
        print("Chronological comparison not assessed: insufficient distinct valid timestamps.")
    else:
        split_ts = ts[valid].sort_values().iloc[len(ts[valid]) // 2]
        print(f"Chronological split at {split_ts} (UTC); this is not a held-out test set.")
        for dimension in ("sportsbook", "regime", "sport"):
            if dimension not in execution_rows.columns:
                continue
            earlier = execution_rows.loc[valid & (ts < split_ts)]
            later = execution_rows.loc[valid & (ts >= split_ts)]
            if earlier.empty or later.empty:
                print(f"{dimension}: no valid two-sided split")
                continue
            early_table = build_segment_evaluation(earlier, dimension, cfg=segment_cfg)
            late_table = build_segment_evaluation(later, dimension, cfg=segment_cfg)
            assert int(early_table.execution_count.sum() + late_table.execution_count.sum()) == int(
                valid.sum()
            )
            fields = ["segment", "execution_count", "realized_roi", "meets_min_executions"]
            paired = early_table[fields].merge(
                late_table[fields], on="segment", how="outer", suffixes=("_earlier", "_later")
            )
            paired["roi_change_later_minus_earlier"] = (
                paired["realized_roi_later"] - paired["realized_roi_earlier"]
            )
            print(f"{dimension}: chronological segment comparison (top earlier-period counts)")
            display(paired.sort_values("execution_count_earlier", ascending=False).head(15))

## Group-aware chronological validation — leakage audit

This section constructs disjoint **earlier** and **later** execution populations using the recorded execution timestamp (`created_at_et`) and the research opportunity grouping key. Opportunity groups that span the chronological cutoff are quarantined rather than assigned to both populations. Rows with missing execution timestamps are excluded and explicitly counted. The audit reconciles every execution to one of these categories.

This is an **exploratory temporal stability diagnostic**, not a prospective or untouched holdout: the dataset and candidate filters have already been inspected. Repeated executions and overlapping saved-filter labels can still introduce dependence within each period. No production filter change is authorized by these comparisons.

In [ ]:
temporal_cfg = TemporalValidationConfig(
    timestamp_column="created_at_et",
    group_column="research_opportunity_key",
    earlier_fraction=0.5,
)
try:
    earlier_executions, later_executions, temporal_audit = build_group_aware_temporal_split(
        execution_rows, cfg=temporal_cfg
    )
except (KeyError, ValueError) as exc:
    print(f"Group-aware chronological validation not assessed: {exc}")
else:
    display(temporal_audit)
    assert bool(temporal_audit.loc[0, "reconciles"])
    assert temporal_audit.loc[0, "shared_opportunity_groups"] == 0
    for dimension in ("regime", "sportsbook", "sport"):
        if dimension not in execution_rows.columns:
            print(f"{dimension}: unavailable in source data")
            continue
        early = build_segment_evaluation(earlier_executions, dimension, cfg=segment_cfg)
        late = build_segment_evaluation(later_executions, dimension, cfg=segment_cfg)
        columns = ["segment", "execution_count", "realized_roi", "meets_min_executions"]
        comparison = early[columns].merge(
            late[columns], on="segment", how="outer", suffixes=("_earlier", "_later")
        )
        comparison["roi_change_later_minus_earlier"] = (
            comparison["realized_roi_later"] - comparison["realized_roi_earlier"]
        )
        print(f"{dimension}: group-disjoint chronological comparison (descriptive)")
        display(comparison.sort_values("execution_count_later", ascending=False).head(15))

## Exploratory research — Canonical filter evaluation tables
These reusable, regime-aware evaluation tables support the Notebook 02 research workflow. Keep execution-level versus opportunity-level semantics explicit, and do not treat production tags as evidence of strategy validity. These outputs are retained for historical comparability.

In [ ]:
# ------------------------------------------------------------------------------
# Canonical filter evaluation tables
# ------------------------------------------------------------------------------

eval_cfg = FilterEvalConfig()

filter_eval_all = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime=None,
)

filter_eval_exchange = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime="exchange",
)

filter_eval_sportsbook = build_filter_evaluation_table(
    df_filtered,
    cfg=eval_cfg,
    regime="sportsbook",
)

print("All filters:")
display(filter_eval_all.head(20))

print("Exchange filters:")
display(filter_eval_exchange.head(20))

print("Sportsbook filters:")
display(filter_eval_sportsbook.head(20))

In [ ]:
# ------------------------------------------------------------------------------
# Ranked views
# ------------------------------------------------------------------------------

rank_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "total_stake",
    "total_profit",
    "total_actual_roi",
    "total_ev",
    "total_ev_roi",
    "profit_minus_ev",
    "ev_realization_gap",
    "edge_capture_ratio",
    "capture_band",
    "avg_clv",
    "max_drawdown",
    "roi_ci_low",
    "roi_ci_high",
    "duplicate_ratio",
    "sample_tier",
]

rank_cols = [c for c in rank_cols if c in filter_eval_all.columns]

exchange_ranked = (
    filter_eval_exchange[rank_cols]
    .sort_values(
        ["is_prod_filter", "total_actual_roi", "total_profit"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

sportsbook_ranked = (
    filter_eval_sportsbook[rank_cols]
    .sort_values(
        ["is_prod_filter", "total_actual_roi", "total_profit"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

print("Ranked exchange filters")
display(exchange_ranked.head(25))

print("Ranked sportsbook filters")
display(sportsbook_ranked.head(25))

## Exploratory research — Exchange capacity diagnostics
This section uses liquidity buckets as imperfect proxies for executable size. Sparse bucket coverage, quoted versus filled amounts, and filter overlap limit inference. The analysis is retained here for reproducibility.

In [ ]:
# ------------------------------------------------------------------------------
# Exchange capacity diagnostics
# ------------------------------------------------------------------------------

exchange_capacity = build_exchange_capacity_table(df_filtered)

display(exchange_capacity.head(25))

## Exploratory research — Exchange performance and capacity scorecard
Combines performance, production tags, duplicate diagnostics, and liquidity proxies. This is an exploratory prioritization view, **not** an automatic promotion or deployment rule. This analysis belongs in the filter-evaluation workflow (Notebook 02).

In [ ]:
# ------------------------------------------------------------------------------
# Exchange scorecard
# ------------------------------------------------------------------------------

scorecard_cfg = ExchangeScorecardConfig()

exchange_scorecard = build_exchange_scorecard(
    filter_eval_exchange=filter_eval_exchange,
    exchange_capacity=exchange_capacity,
    cfg=scorecard_cfg,
)

scorecard_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "total_stake",
    "total_profit",
    "total_actual_roi",
    "total_ev",
    "total_ev_roi",
    "profit_minus_ev",
    "ev_realization_gap",
    "edge_capture_ratio",
    "capture_band",
    "avg_clv",
    "max_drawdown",
    "roi_ci_low",
    "roi_ci_high",
    "duplicate_ratio",
    "sample_tier",
    "liquidity_coverage",
    "avg_liquidity",
    "median_liquidity",
    "capacity_estimate",
    "capacity_ratio",
    "liquidity_weighted_ev",
    "capacity_to_stake_ratio",
]

scorecard_cols = [c for c in scorecard_cols if c in exchange_scorecard.columns]

exchange_scorecard = exchange_scorecard[scorecard_cols].reset_index(drop=True)

display(exchange_scorecard.head(25))

## Historical filter evidence review — candidates and validation requirements

This report consolidates the **canonical filter evaluation** and **exchange liquidity-coverage diagnostics** into a conservative review table. It is not a production-filter recommendation or a causal comparison between saved filter settings. Filter labels can overlap, a filter name does not guarantee a single execution regime, and repeated opportunity IDs create dependence between executions.

A positive historical ROI or favorable confidence interval is only a **candidate for further validation**. The interval calculations are inherited from the existing evaluation module and have **not** been verified here as opportunity-group-aware. Liquidity coverage describes observability, not guaranteed fill size. Missing CLV, unavailable market fields, unknown historical filter configuration, and snapshot recency remain limitations.

The table retains all evaluated rows and explicitly documents required follow-up. No filter is automatically approved for activation, retirement, or threshold changes. The next research stage should validate candidates on later non-overlapping periods with group-aware uncertainty, overlap controls, and execution-quality checks.

In [ ]:
# Consolidate regime-specific canonical results without double-counting a combined view.
review_inputs = pd.concat([filter_eval_exchange, filter_eval_sportsbook], ignore_index=True)
review_cfg = CandidateReviewConfig(min_executions=100, min_liquidity_coverage=0.8)
candidate_review = build_candidate_review(review_inputs, capacity=exchange_capacity, cfg=review_cfg)
assert len(candidate_review) == len(review_inputs)
assert not candidate_review["production_change_authorized"].any()

print("Historical evidence-review classifications (not deployment decisions):")
display(
    candidate_review["review_classification"]
    .value_counts(dropna=False)
    .to_frame("filter_regime_rows")
)
print("Review table — production filters first, then execution count:")
review_columns = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "total_stake",
    "total_actual_roi",
    "total_ev_roi",
    "roi_ci_low",
    "roi_ci_high",
    "liquidity_coverage",
    "meets_review_count",
    "roi_interval_available",
    "roi_interval_positive",
    "exchange_liquidity_adequately_observed",
    "review_classification",
    "required_follow_up",
    "production_change_authorized",
]
review_columns = [column for column in review_columns if column in candidate_review.columns]
review_sort = [column for column in ("is_prod_filter", "bet_count") if column in candidate_review]
review_display = (
    candidate_review.sort_values(review_sort, ascending=[False] * len(review_sort))
    if review_sort
    else candidate_review
)
display(review_display[review_columns].head(30))
print("Historical data only; no production change is authorized by this report.")

## Exploratory research — Exchange EV versus realized ROI plot
Expected and realized ROI are different estimands. Differences may reflect sampling variance, reference-price calibration, selection, or execution; they cannot be attributed to execution quality alone. Preserve this plot for Notebook 02.

In [ ]:
# ------------------------------------------------------------------------------
# Exchange filters: EV ROI vs realized ROI
# ------------------------------------------------------------------------------

plot_df = filter_eval_exchange.copy()

plot_df = plot_df[
    plot_df["total_ev_roi"].notna()
    & plot_df["total_actual_roi"].notna()
    & plot_df["bet_count"].notna()
].copy()

plt.figure(figsize=(10, 7))

prod_mask = (
    plot_df["is_prod_filter"].fillna(False).astype(bool)
    if "is_prod_filter" in plot_df.columns
    else pd.Series(False, index=plot_df.index)
)

# Non-prod first
plt.scatter(
    plot_df.loc[~prod_mask, "total_ev_roi"],
    plot_df.loc[~prod_mask, "total_actual_roi"],
    s=np.clip(plot_df.loc[~prod_mask, "bet_count"].fillna(0) * 2, 20, 400),
    alpha=0.55,
    label="test filters",
)

# Prod on top
plt.scatter(
    plot_df.loc[prod_mask, "total_ev_roi"],
    plot_df.loc[prod_mask, "total_actual_roi"],
    s=np.clip(plot_df.loc[prod_mask, "bet_count"].fillna(0) * 2, 20, 400),
    alpha=0.9,
    marker="D",
    label="prod filters",
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

for _, row in plot_df.iterrows():
    if row["bet_count"] >= 100:
        plt.annotate(
            row["saved_filter_names"],
            (row["total_ev_roi"], row["total_actual_roi"]),
            fontsize=8,
            alpha=0.8,
        )

plt.xlabel("Total EV ROI")
plt.ylabel("Total Realized ROI")
plt.title("Exchange Filters: EV ROI vs Realized ROI")
plt.legend()
plt.show()

## Exploratory research — Filter-level duplicate-ratio diagnostics
These diagnostics describe repeated signatures and possible opportunity overlap, not confirmed duplicate ingestion. Basic duplicate integrity is a Notebook 01 concern; performance-adjusted duplicate analysis is Notebook 02 research.

In [ ]:
# ------------------------------------------------------------------------------
# Duplicate ratio diagnostics
# ------------------------------------------------------------------------------

dup_diag_cols = [
    "saved_filter_names",
    "regime",
    "is_prod_filter",
    "bet_count",
    "n_unique_opportunities",
    "duplicate_ratio",
    "total_actual_roi",
    "total_ev_roi",
    "edge_capture_ratio",
    "capture_band",
]

dup_diag_cols = [c for c in dup_diag_cols if c in filter_eval_all.columns]

dup_diag = filter_eval_all[dup_diag_cols].copy()
dup_diag = dup_diag.sort_values("duplicate_ratio", ascending=False).reset_index(drop=True)

display(dup_diag.head(25))

## Filter-level ROI and modeled EV

The first plot uses canonical filter-evaluation metrics rather than a legacy notebook-only summary dataframe. Marker size indicates sample volume. Differences between realized and modeled ROI should not be interpreted as execution slippage without additional evidence.

In [ ]:
# Canonical filter-level realized ROI versus modeled EV ROI.
# Both measures are expressed as percentages; marker area scales with bet count.
plot_df = filter_eval_all.dropna(subset=["total_actual_roi", "total_ev_roi"]).copy()
plot_df = plot_df.sort_values("bet_count", ascending=False).head(30)
fig, ax = plt.subplots(figsize=(11, 7))
if not plot_df.empty:
    sizes = 30 + 300 * (plot_df["bet_count"] / max(plot_df["bet_count"].max(), 1))
    ax.scatter(
        100 * plot_df["total_ev_roi"], 100 * plot_df["total_actual_roi"], s=sizes, alpha=0.65
    )
    lo = min(0, (100 * plot_df[["total_ev_roi", "total_actual_roi"]]).min().min())
    hi = max(0, (100 * plot_df[["total_ev_roi", "total_actual_roi"]]).max().max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1, color="gray")
else:
    ax.text(0.5, 0.5, "No filters with both ROI measures", ha="center", transform=ax.transAxes)
ax.set(
    xlabel="Modeled EV ROI (%)",
    ylabel="Realized ROI (%)",
    title="Filter-level realized versus modeled ROI (largest samples)",
)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## EV bucket diagnostics

Compare modeled EV with realized outcomes across EV buckets. Small samples and multiple comparisons can generate unstable patterns; use these charts to prioritize research, not to automatically alter live filters.

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=0.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=100,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name=None,
    cfg=cfg,
    x_bucket="theoretical_ev",
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=50,
    min_bets_per_bucket=50,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
    x_bucket="theoretical_ev",
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=30,
    min_bets_per_bucket=25,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=cfg,
    x_bucket="theoretical_ev",
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=25,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name=None,
    cfg=cfg,
    x_bucket="group_col",
    group_col="time_to_event_bucket",
    only_non_null_group=True,
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=25,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
    x_bucket="group_col",
    group_col="time_to_event_bucket",
    only_non_null_group=True,
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=25,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=cfg,
    x_bucket="group_col",
    group_col="time_to_event_bucket",
    only_non_null_group=True,
)

## Liquidity-stratified performance

Exchange liquidity buckets have incomplete historical coverage. These charts condition on available liquidity metadata and should not be generalized to missing-liquidity bets without checking selection effects.

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=100,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name=None,
    cfg=cfg,
    x_bucket="group_col",
    group_col="liquidity_bucket",
    only_non_null_group=True,
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=25,
    min_bets_per_bucket=50,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot(
    df=df_filtered,
    filter_name="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
    x_bucket="group_col",
    group_col="liquidity_bucket",
    only_non_null_group=True,
)

In [ ]:
cfg = CombinedEVPlotConfig(
    bin_size=0.5,
    min_x_pct=2.0,
    max_x_pct=20.0,
    y_lim_pct=20,
    min_bets_per_bucket=25,
    show_ev_dollar_labels=True,
    clv_weight="stake",  # "stake" or "equal"
    bucket_edge="left",  # "left" or "right"
    bar_color_by="profit",  # so colors align with profit
    show_theoretical_ev_line=True,
)

combined_ev_summary_plot_faceted_by_group(
    df=df_filtered,
    filter_name=None,
    facet_col="liquidity_bucket",
    cfg=cfg,
    x_bucket="theoretical_ev",
    only_non_null_facet=True,
)

## Cumulative profit and venue/sport decomposition

Executed-bet profit curves describe historical outcomes, not necessarily independently executable opportunities. Filter-level series may overlap.

In [ ]:
# Execution-level cumulative profit for the highest-profit saved filters.
# Filter labels may overlap; this is a descriptive comparison, not an additive portfolio.
required = {"created_at_et", "saved_filter_names", "profit"}
missing = required - set(df_filtered.columns)
if missing:
    print(f"Skipping cumulative-profit plot; missing columns: {sorted(missing)}")
else:
    top_names = (
        df_filtered.groupby("saved_filter_names", dropna=False)["profit"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index
    )
    fig, ax = plt.subplots(figsize=(12, 6))
    for name in top_names:
        sub = df_filtered.loc[df_filtered["saved_filter_names"].eq(name)].copy()
        sub["created_at_et"] = pd.to_datetime(sub["created_at_et"], errors="coerce")
        sub = sub.dropna(subset=["created_at_et"]).sort_values("created_at_et")
        if not sub.empty:
            ax.plot(sub["created_at_et"], sub["profit"].cumsum(), label=str(name), alpha=0.85)
    ax.set(
        title="Cumulative realized profit by saved filter (top 10 by total profit)",
        xlabel="Bet timestamp (ET)",
        ylabel="Cumulative profit ($)",
    )
    ax.axhline(0, color="gray", linewidth=1)
    ax.legend(fontsize=7, loc="best")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
cfg = CumProfitByGroupPlotConfig(resample="W")

plot_cum_profit_by_group(
    df=df_filtered, group_col="sportsbook", filters=None, global_view=True, cfg=cfg
)

In [ ]:
plot_cum_profit_by_group(
    df=df_filtered,
    group_col="sportsbook",
    filters="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

In [ ]:
plot_cum_profit_by_group(
    df=df_filtered,
    group_col="sportsbook",
    filters="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

In [ ]:
cfg = ProfitByGroupPlotConfig(min_bets_per_group=50, top_k=15, combine_filters=False)

plot_profit_by_group_for_filters(
    df=df_filtered, group_col="sport", filters=None, global_view=True, cfg=cfg
)

In [ ]:
plot_profit_by_group_for_filters(
    df=df_filtered,
    group_col="sport",
    filters="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

In [ ]:
plot_profit_by_group_for_filters(
    df=df_filtered,
    group_col="sport",
    filters="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

## Activity timing and filter-specific volume

Time-series activity plots are useful for detecting changing execution mix and filter deployment. They do not establish a causal time-of-day or day-of-week edge.

In [ ]:
activity_volume_cfg = BetCountOverTimePlotConfig(resample="W")

plot_bet_count_over_time(
    df=df_filtered,
    filters="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=activity_volume_cfg,
)

In [ ]:
plot_bet_count_over_time(
    df=df_filtered,
    filters="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=activity_volume_cfg,
)

In [ ]:
cfg = AvgBetsByTimeBucketPlotConfig(bucket="dow")

plot_avg_bets_by_time_bucket(df=df_filtered, filters=None, global_view=True, cfg=cfg)

In [ ]:
plot_avg_bets_by_time_bucket(
    df=df_filtered,
    filters="PROD - Exchanges: Core Exchange Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

In [ ]:
plot_avg_bets_by_time_bucket(
    df=df_filtered,
    filters="PROD - Offshore Books: Core & Conservative (Low Risk, Slow Build)",
    cfg=cfg,
)

In [ ]:
# ------------------------------------------------------------------------------
# Optional artifact writes
# ------------------------------------------------------------------------------

if WRITE_ARTIFACTS:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    filter_eval_all.to_parquet(
        ARTIFACT_DIR / f"filter-eval-all-{AS_OF_DATE}.parquet",
        index=False,
    )
    filter_eval_exchange.to_parquet(
        ARTIFACT_DIR / f"filter-eval-exchange-{AS_OF_DATE}.parquet",
        index=False,
    )
    filter_eval_sportsbook.to_parquet(
        ARTIFACT_DIR / f"filter-eval-sportsbook-{AS_OF_DATE}.parquet",
        index=False,
    )
    exchange_capacity.to_parquet(
        ARTIFACT_DIR / f"exchange-capacity-{AS_OF_DATE}.parquet",
        index=False,
    )
    exchange_scorecard.to_parquet(
        ARTIFACT_DIR / f"exchange-scorecard-{AS_OF_DATE}.parquet",
        index=False,
    )

    print("Wrote regime-aware, prod-aware evaluation artifacts.")
else:
    print("WRITE_ARTIFACTS=False; no parquet artifacts written.")